# Testing the draft rug plot (`team/new_graphics/rug.py`)

This notebook exercises `make_rug()`, the draft implementation of the new rug plot
type from the approved Task 1E prototype. It is **not** wired into `.plot()` yet —
this is for visually checking the draft before integration.

**How to run:** open this notebook from the `team/new_graphics/new_graphics_demos/`
folder using the `symbulate` conda environment, then Run All.

**What to look for in every plot:**
- One thin vertical tick per simulated value, rising from the bottom of the axes
- Stacked/repeated values read as darker ticks (alpha 0.5)
- No y-axis ticks and no left spine; only the bottom axis line remains
- Faint *vertical* reference gridlines instead of horizontal ones
- x-axis always reads "Value"; title reads "Rug Plot"

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# This notebook lives in <repo>/team/new_graphics/new_graphics_demos/.
REPO_ROOT = Path.cwd().parents[2]

# Make `import symbulate` find the dev copy in this repo (not an older
# installed copy in site-packages), and `from rug import ...` find the
# draft module one folder up.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(Path.cwd().parent))

# The draft helper and its per-plot-type constants
from rug import make_rug, RUG_ALPHA, RUG_TICK_HEIGHT

# Apply the global style file (Okabe-Ito cycle, grid, spines, fonts).
# Not yet wired into the package, so we apply it by hand here.
STYLE_PATH = REPO_ROOT / "symbulate" / "symbulate.mplstyle"
plt.style.use(STYLE_PATH)

rng = np.random.default_rng(7)
print(f"RUG_ALPHA = {RUG_ALPHA}, RUG_TICK_HEIGHT = {RUG_TICK_HEIGHT}")

## 1. Reproduce the approved prototype

This should match the prototype image: ~60 draws from Normal(0, 1), light
sky-blue ticks along the bottom. The short, wide figure matches the prototype
image and is demo-only — the final `.plot()` integration keeps the global
default figure size.

In [ ]:
values = rng.normal(0, 1, 60)

plt.figure(figsize=(9, 2))  # demo-only, to match the prototype image
ax = plt.gca()
make_rug(values, ax, "#56B4E9")  # Okabe-Ito sky blue, as in the prototype
plt.show()  # title and axis labels are set automatically by make_rug

## 2. Options: default figure size, custom alpha

The same rug at the package's standard figure size (how it will actually
appear from `.plot()`), then with an alpha override.

In [ ]:
ax = plt.gca()
make_rug(values, ax, "#56B4E9")
plt.show()

In [ ]:
# Fully opaque ticks -- stacked values no longer read as darker
ax = plt.gca()
make_rug(values, ax, "#56B4E9", alpha=1.0)
plt.show()

## 3. Overlay behavior — mimicking two `.plot()` calls

In the final integration, each `.plot()` call takes its color from
`get_next_color(ax)`. Two overlaid rugs should come out in distinct Okabe-Ito
hues, and a legend should appear **automatically in the top right** as soon as
a second rug lands on the axes, auto-named "Variable 1", "Variable 2", ...
A single rug gets no legend.

(Importing `symbulate` currently switches the matplotlib style back to the old
seaborn/ggplot stylesheet — a Phase 2 item — so we re-apply ours right after.)

In [ ]:
from symbulate.plot import get_next_color

plt.style.use(STYLE_PATH)  # re-apply: importing symbulate overrides the style

sample_a = rng.normal(0, 1, 60)
sample_b = rng.normal(1.5, 0.5, 60)

ax = plt.gca()
make_rug(sample_a, ax, get_next_color(ax))
make_rug(sample_b, ax, get_next_color(ax))  # legend appears on this call
plt.show()

### Custom legend labels

`label=` overrides the automatic "Variable k" names.

In [ ]:
ax = plt.gca()
make_rug(rng.normal(0, 1, 60), ax, "#56B4E9", label="Control")
make_rug(rng.normal(1.5, 0.5, 60), ax, "#E69F00", label="Treatment")
plt.show()

## 4. With real symbulate simulation data

Small-n simulations are exactly where the lookup table will default to a rug
plot, so test with `.sim(50)`.

Caveat while testing today: creating `RVResults` currently calls `init_color()`,
which resets the color cycle to tab10 (removed in Phase 2), so we re-apply the
style *after* simulating.

In [ ]:
from symbulate import RV, Normal

sims = RV(Normal(0, 1)).sim(50)

plt.style.use(STYLE_PATH)  # re-apply: RVResults resets the color cycle to tab10

ax = plt.gca()
make_rug(np.asarray(sims.results), ax, "#56B4E9")
plt.show()

## 5. Combined with a density curve — `type=("density", "rug")`

This cell sketches what `.plot(type=("density", "rug"))` will produce: a KDE
density curve estimated from the simulated values, with a short rug of the same
values hugging the x-axis underneath. It should match the approved
density-plus-rug prototype.

The rug is drawn with `standalone=False`: the ticks shrink to
`RUG_COMBINED_TICK_HEIGHT` (4% of the axes height, in axes fractions, so the
density curve's y-scale doesn't affect them), and the y-axis, left spine,
horizontal grid, and title are left to the companion plot type.

There is no `make_density` draft module yet, so the curve half is drawn inline
here using the package's existing `compute_density` helper — when `density.py`
gets drafted, it will own the curve, the baseline at 0, the "Density" y-label,
and the "Density Curve" title.

In [ ]:
from symbulate.plot import compute_density

values = np.asarray(sims.results)

ax = plt.gca()

# The rug half: short ticks on the x-axis, axes furniture untouched.
make_rug(values, ax, "#56B4E9", standalone=False)

# The density-curve half (stand-in for the future make_density, which
# will own these choices): KDE of the same simulated values, drawn at
# the style guide's density line width (1.8, a future plot.py
# constant), baseline kept at 0, "Density" y-label, "Density Curve"
# title. Okabe-Ito blue, as in the prototype image.
density = compute_density(values)
pad = 0.2 * (values.max() - values.min())
xs = np.linspace(values.min() - pad, values.max() + pad, 1000)
ax.plot(xs, density(xs), color="#0072B2", linewidth=1.8)
ax.set_ylim(bottom=0)
ax.set_ylabel("Density")
ax.set_title("Density Curve")
plt.show()

## Verification checklist

- [ ] Section 1 matches the approved prototype image
- [ ] One tick per value; stacked values read as darker ticks
- [ ] Standalone rug: no y-axis ticks; no left/top/right spines — only the bottom axis line
- [ ] Standalone rug: vertical reference gridlines
- [ ] x-axis always reads "Value"; standalone title reads "Rug Plot"
- [ ] `alpha=` override works
- [ ] Single rug has no legend
- [ ] Legend appears in the top right on the second overlaid rug, auto-named "Variable 1", "Variable 2", ...
- [ ] `label=` overrides the automatic legend names
- [ ] Works on real `.sim()` output, not just numpy arrays
- [ ] `standalone=False` matches the density-plus-rug prototype: short ticks on the x-axis, y-axis/spine/grid untouched
- [ ] Combined ticks keep their height regardless of the density curve's y-scale (axes-fraction transform)